In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix,precision_score, recall_score,f1_score,roc_curve, roc_auc_score, ConfusionMatrixDisplay, make_scorer
from sklearn.model_selection import train_test_split, cross_val_score, cross_validate, GridSearchCV
import time
import copy
import os
from collections import Counter
from sklearn.metrics import log_loss
import pickle

import import_ipynb
import importlib
import functions as fc
importlib.reload(fc)

print(os.getcwd())

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

c:\Users\ameli\OneDrive\Studium\TUWien\SS2026\PrivacyInML\Exercise2


In [3]:
def evaluate_true_class_confidence(input_confidence_scores, y_train, y_test):

    true_labels = np.concatenate([y_train, y_test])
    class_order = np.array([0, 1])

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])

    all_mean = []
    member_mean = []
    nonmember_mean = []
    member_loss_mean = []
    nonmember_loss_mean = []
    member_entropy_mean = []
    nonmember_entropy_mean = []

    for key, scores in input_confidence_scores.items():

        true_class_indices = np.array([np.where(class_order == label)[0][0]for label in true_labels])
        true_confidences = scores[np.arange(len(true_labels)),true_class_indices]
        member_confidence = true_confidences[:len(y_train)]
        nonmember_confidence = true_confidences[len(y_train):]

        ##### LOSS #####
        loss = -np.log(np.clip(true_confidences, 1e-10, 1.0))
        member_loss = loss[:len(y_train)]
        nonmember_loss = loss[len(y_train):]

        # Higher confidence = more likely member
        auc = roc_auc_score(membership,true_confidences)
        # Lower loss = more likely member
        auc_loss = roc_auc_score(membership,-loss)

        ##### ENTROPY #####
        entropy = -np.sum(scores * np.log(np.clip(scores, 1e-10, 1.0)),axis=1)
        member_entropy = entropy[:len(y_train)]
        nonmember_entropy = entropy[len(y_train):]
        # Lower entropy = more confident prediction
        auc_entropy = roc_auc_score(membership,-entropy)

        ##### RESULTS #####
        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Confidence:")
        print("  Overall mean:", true_confidences.mean())
        print("  Member mean:", member_confidence.mean())
        print("  Non-member mean:", nonmember_confidence.mean())
        print("  Confidence gap:",
              member_confidence.mean() - nonmember_confidence.mean())
        print("  MIA AUC:", auc)

        print()
        print("Loss:")
        print("  Overall mean:", loss.mean())
        print("  Member mean:", member_loss.mean())
        print("  Non-member mean:", nonmember_loss.mean())
        print("  Loss gap:",
              nonmember_loss.mean() - member_loss.mean())
        print("  MIA AUC:", auc_loss)

        print()
        print("Entropy:")
        print("  Overall mean:", entropy.mean())
        print("  Member mean:", member_entropy.mean())
        print("  Non-member mean:", nonmember_entropy.mean())
        print("  Entropy gap:",
              nonmember_entropy.mean() - member_entropy.mean())
        print("  MIA AUC:", auc_entropy)

        print()
        print("Confidence:")
        print("  Minimum:", true_confidences.min())
        print("  Maximum:", true_confidences.max())

        print('=========================================\n')

        all_mean.append(true_confidences.mean())
        member_mean.append(member_confidence.mean())
        nonmember_mean.append(nonmember_confidence.mean())
        member_loss_mean.append(member_loss.mean())
        nonmember_loss_mean.append(nonmember_loss.mean())
        member_entropy_mean.append(member_entropy.mean())
        nonmember_entropy_mean.append(nonmember_entropy.mean())


    # ======================================================
    # ACROSS ALL MODELS
    # ======================================================
    print("=== Across all models ===")
    print("Average overall confidence:", np.mean(all_mean))
    print("Average member confidence:", np.mean(member_mean))
    print("Average non-member confidence:", np.mean(nonmember_mean))
    print()
    print("Average member loss:", np.mean(member_loss_mean))
    print("Average non-member loss:", np.mean(nonmember_loss_mean))
    print()
    print("Average member entropy:", np.mean(member_entropy_mean))
    print("Average non-member entropy:", np.mean(nonmember_entropy_mean))

In [4]:
def evaluate_true_max_confidence(input_confidence_scores, y_train, y_test):

    true_labels = np.concatenate([y_train, y_test])
    class_order = np.array([0, 1])

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])

    all_mean = []
    member_mean = []
    nonmember_mean = []
    member_loss_mean = []
    nonmember_loss_mean = []
    member_entropy_mean = []
    nonmember_entropy_mean = []

    for key, scores in input_confidence_scores.items():

        true_confidences = scores.max(axis=1)
        member_confidence = true_confidences[:len(y_train)]
        nonmember_confidence = true_confidences[len(y_train):]

        ##### LOSS #####
        loss = -np.log(np.clip(true_confidences, 1e-10, 1.0))
        member_loss = loss[:len(y_train)]
        nonmember_loss = loss[len(y_train):]

        # Higher confidence = more likely member
        auc = roc_auc_score(membership,true_confidences)
        # Lower loss = more likely member
        auc_loss = roc_auc_score(membership,-loss)

        ##### ENTROPY #####
        entropy = -np.sum(scores * np.log(np.clip(scores, 1e-10, 1.0)),axis=1)
        member_entropy = entropy[:len(y_train)]
        nonmember_entropy = entropy[len(y_train):]
        # Lower entropy = more confident prediction
        auc_entropy = roc_auc_score(membership,-entropy)

        ##### RESULTS #####
        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Confidence:")
        print("  Overall mean:", true_confidences.mean())
        print("  Member mean:", member_confidence.mean())
        print("  Non-member mean:", nonmember_confidence.mean())
        print("  Confidence gap:",
              member_confidence.mean() - nonmember_confidence.mean())
        print("  MIA AUC:", auc)

        print()
        print("Loss:")
        print("  Overall mean:", loss.mean())
        print("  Member mean:", member_loss.mean())
        print("  Non-member mean:", nonmember_loss.mean())
        print("  Loss gap:",
              nonmember_loss.mean() - member_loss.mean())
        print("  MIA AUC:", auc_loss)

        print()
        print("Entropy:")
        print("  Overall mean:", entropy.mean())
        print("  Member mean:", member_entropy.mean())
        print("  Non-member mean:", nonmember_entropy.mean())
        print("  Entropy gap:",
              nonmember_entropy.mean() - member_entropy.mean())
        print("  MIA AUC:", auc_entropy)

        print()
        print("Confidence:")
        print("  Minimum:", true_confidences.min())
        print("  Maximum:", true_confidences.max())

        print('=========================================\n')

        all_mean.append(true_confidences.mean())
        member_mean.append(member_confidence.mean())
        nonmember_mean.append(nonmember_confidence.mean())
        member_loss_mean.append(member_loss.mean())
        nonmember_loss_mean.append(nonmember_loss.mean())
        member_entropy_mean.append(member_entropy.mean())
        nonmember_entropy_mean.append(nonmember_entropy.mean())

    # ======================================================
    # ACROSS ALL MODELS
    # ======================================================
    print("=== Across all models ===")
    print("Average overall confidence:", np.mean(all_mean))
    print("Average member confidence:", np.mean(member_mean))
    print("Average non-member confidence:", np.mean(nonmember_mean))
    print()
    print("Average member loss:", np.mean(member_loss_mean))
    print("Average non-member loss:", np.mean(nonmember_loss_mean))
    print()
    print("Average member entropy:", np.mean(member_entropy_mean))
    print("Average non-member entropy:", np.mean(nonmember_entropy_mean))

In [5]:
def evaluate_threshold_attack(
    input_confidence_scores,
    y_train,
    y_test,
    threshold
):

    membership = np.concatenate([np.ones(len(y_train)),np.zeros(len(y_test))])
    for key, scores in input_confidence_scores.items():

        # Use the highest predicted probability
        confidence = scores.max(axis=1)

        # Predict membership based on threshold
        attack_labels = np.array([1 if conf > threshold else 0 for conf in confidence])

        print('-----------------')
        print(f"Model {key}")
        print('-----------------')

        print("Threshold:", threshold)

        print("Predicted members:",np.sum(attack_labels == 1))
        print("Predicted non-members:",np.sum(attack_labels == 0))
        print("Actual members:",np.sum(membership == 1))
        print("Actual non-members:",np.sum(membership == 0))

        print("Accuracy:",accuracy_score(membership, attack_labels))
        print("Confusion matrix:")

        cm = confusion_matrix(membership,attack_labels)
        print(cm)
        print('=========================================\n')

In [6]:
cc_train = pd.read_csv("datasets/creditcard_train.csv")
cc_test = pd.read_csv("datasets/creditcard_test.csv")

cc_train = cc_train.drop("ID", axis = 1)
cc_test = cc_test.drop("ID", axis = 1)

def get_dummies_all(X_train, X_test):
    """
    Converts all categorical variables in a DataFrame into dummy (one-hot encoded) variables.
    """
    categorical_cols = ["MARRIAGE", "SEX", "EDUCATION"]  # Select categorical columns
    X_train_encoded = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
    X_test_encoded = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)
    return X_train_encoded, X_test_encoded

X_train = cc_train.drop('default.payment.next.month', axis = 1)
X_test = cc_test.drop('default.payment.next.month', axis = 1)
y_train = cc_train['default.payment.next.month']
y_test = cc_test['default.payment.next.month']

X_train, X_test = get_dummies_all(X_train, X_test)

----------------------------------------------------------------------------------------------
-------------------------------------- INPUT PERTUBATION -------------------------------------
----------------------------------------------------------------------------------------------

In [7]:
with open("input_confidence_scores_creditcard.pkl", "rb") as f:
    input_confidence_scores = pickle.load(f)

In [8]:
evaluate_true_class_confidence(input_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(input_confidence_scores, y_train, y_test)

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.6580340742874106
  Member mean: 0.6577736380968429
  Non-member mean: 0.6590758190496816
  Confidence gap: -0.0013021809528387385
  MIA AUC: 0.5027793472222223

Loss:
  Overall mean: 0.523594634280134
  Member mean: 0.5242628505162344
  Non-member mean: 0.5209217693357325
  Loss gap: -0.0033410811805019147
  MIA AUC: 0.5027793472222223

Entropy:
  Overall mean: 0.526081560279759
  Member mean: 0.5259803804001015
  Non-member mean: 0.5264862797983895
  Entropy gap: 0.0005058993982879345
  MIA AUC: 0.5051175729166667

Confidence:
  Minimum: 0.16820479576030606
  Maximum: 0.8459087448131746

-----------------
Model 1
-----------------
Confidence:
  Overall mean: 0.6553213108455671
  Member mean: 0.6551054636292488
  Non-member mean: 0.6561846997108404
  Confidence gap: -0.0010792360815915503
  MIA AUC: 0.5003532777777777

Loss:
  Overall mean: 0.5428954324595783
  Member mean: 0.5436282200291411
  Non-member mean:

In [9]:
evaluate_threshold_attack(input_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 0
Predicted non-members: 30000
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.2
Confusion matrix:
[[ 6000     0]
 [24000     0]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 1
Predicted non-members: 29999
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.20003333333333334
Confusion matrix:
[[ 6000     0]
 [23999     1]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 75
Predicted non-members: 29925
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.2013
Confusion matrix:
[[ 5982    18]
 [23943    57]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Predicted members: 89
Predicted non-members: 29911
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.2017
Confusion matrix:
[[ 5981    19]
 [23930    70]]

-----------------
Model 30
----------

----------------------------------------------------------------------------------------------
------------------------------------- OUTPUT PERTUBATION -------------------------------------
----------------------------------------------------------------------------------------------

In [10]:
with open("output_confidence_scores_creditcard.pkl", "rb") as f:
    output_confidence_scores = pickle.load(f)

In [11]:
evaluate_true_class_confidence(output_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(output_confidence_scores, y_train, y_test)

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.706322654757363
  Member mean: 0.7063037632979451
  Non-member mean: 0.7063982205950349
  Confidence gap: -9.445729708978945e-05
  MIA AUC: 0.4980556458333333

Loss:
  Overall mean: 0.4641432140348163
  Member mean: 0.46446647505286115
  Non-member mean: 0.4628501699626366
  Loss gap: -0.0016163050902245346
  MIA AUC: 0.4980556458333333

Entropy:
  Overall mean: 0.4645618279896817
  Member mean: 0.46466184587676906
  Non-member mean: 0.46416175644133256
  Entropy gap: -0.0005000894354365015
  MIA AUC: 0.49860922569444444

Confidence:
  Minimum: 0.0013048863699565121
  Maximum: 0.9999999999984461

-----------------
Model 1
-----------------
Confidence:
  Overall mean: 0.7063596249655987
  Member mean: 0.7063398268074897
  Non-member mean: 0.7064388175980341
  Confidence gap: -9.89907905443177e-05
  MIA AUC: 0.49806159027777774

Loss:
  Overall mean: 0.46413542764471044
  Member mean: 0.4644649089837159
  Non-mem

In [12]:
evaluate_threshold_attack(output_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 451
Predicted non-members: 29549
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.2089
Confusion matrix:
[[ 5908    92]
 [23641   359]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 456
Predicted non-members: 29544
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.209
Confusion matrix:
[[ 5907    93]
 [23637   363]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 452
Predicted non-members: 29548
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.20906666666666668
Confusion matrix:
[[ 5910    90]
 [23638   362]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Predicted members: 452
Predicted non-members: 29548
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.20906666666666668
Confusion matrix:
[[ 5910    90]
 [23638   362]]

----------------

----------------------------------------------------------------------------------------------
------------------------------------ INTERNAL PERTUBATION ------------------------------------
----------------------------------------------------------------------------------------------

In [13]:
with open("internal_confidence_scores_creditcard.pkl", "rb") as f:
    internal_confidence_scores = pickle.load(f)

In [14]:
evaluate_true_class_confidence(internal_confidence_scores, y_train, y_test)
evaluate_true_max_confidence(internal_confidence_scores, y_train, y_test)

-----------------
Model 0.1
-----------------
Confidence:
  Overall mean: 0.6262928264061386
  Member mean: 0.6262775967307812
  Non-member mean: 0.626353745107568
  Confidence gap: -7.614837678682207e-05
  MIA AUC: 0.5019283333333333

Loss:
  Overall mean: 7.386414173524312
  Member mean: 7.37462825593091
  Non-member mean: 7.433557843897916
  Loss gap: 0.05892958796700665
  MIA AUC: 0.5012119861111112

Entropy:
  Overall mean: 0.01596169018809949
  Member mean: 0.015825352723258735
  Non-member mean: 0.016507040047462538
  Entropy gap: 0.000681687324203803
  MIA AUC: 0.4940103819444444

Confidence:
  Minimum: 0.0
  Maximum: 1.0

-----------------
Model 1
-----------------
Confidence:
  Overall mean: 0.7029209431826275
  Member mean: 0.7037120055497504
  Non-member mean: 0.699756693714136
  Confidence gap: 0.003955311835614483
  MIA AUC: 0.4989554340277778

Loss:
  Overall mean: 4.913561307192665
  Member mean: 4.90584049574661
  Non-member mean: 4.944444552976879
  Loss gap: 0.038604

In [16]:
evaluate_threshold_attack(internal_confidence_scores,y_train,y_test,threshold=0.970489258446794)

-----------------
Model 0.1
-----------------
Threshold: 0.970489258446794
Predicted members: 28945
Predicted non-members: 1055
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.7791
Confusion matrix:
[[  214  5786]
 [  841 23159]]

-----------------
Model 1
-----------------
Threshold: 0.970489258446794
Predicted members: 28153
Predicted non-members: 1847
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.7637666666666667
Confusion matrix:
[[  380  5620]
 [ 1467 22533]]

-----------------
Model 5
-----------------
Threshold: 0.970489258446794
Predicted members: 13950
Predicted non-members: 16050
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.4784
Confusion matrix:
[[ 3201  2799]
 [12849 11151]]

-----------------
Model 10
-----------------
Threshold: 0.970489258446794
Predicted members: 14196
Predicted non-members: 15804
Actual members: 24000
Actual non-members: 6000
Accuracy: 0.4822
Confusion matrix:
[[ 3135  2865]
 [12669 11331]]

-----------------
Model